In [17]:
import pandas as pd

LEAK_THRESHOLD = 300
MIN_OCCURRENCES = 3
MONTHLY_WINDOW = (25, 35)
AMOUNT_TOLERANCE = 50

CANCEL_DIFFICULTY = {
    "gym": ("hard", "Requires in person visit."),
    "netflix": ("medium", "App cancellation works but buried under 3 menus."),
    "spotify": ("easy", "One-click in account settings."),
    "hotstar": ("medium", "Web only, not in app."),
    "amazon": ("hard", "Prime cancellation requires confirming twice + retention screen."),
}

data = {
    "date": [
        "2024-01-01", "2024-02-01", "2024-03-01",
        "2024-01-05", "2024-02-05", "2024-03-05",
        "2024-01-10", "2024-02-10", "2024-03-10",
        "2024-03-15", "2024-03-18"
    ],
    "description": [
        "Netflix", "Netflix", "Netflix",
        "Spotify", "Spotify", "Spotify",
        "Gym", "Gym", "Gym",
        "Swiggy", "Amazon"
    ],
    "amount": [
        -499, -499, -499,
        -119, -119, -119,
        -999, -999, -999,
        -450, -1499
    ]
}

df = pd.DataFrame(data)
df["date"] = pd.to_datetime(df["date"])
df["amount"] = df["amount"].astype(float)
df = df[df["amount"] < 0].copy()

def is_monthly(group):
    group = group.sort_values("date")
    gaps = group["date"].diff().dropna().dt.days
    return gaps.between(*MONTHLY_WINDOW).all()

def is_consistent_amount(group):
    return group["amount"].std() < AMOUNT_TOLERANCE

subscriptions = []

for merchant, group in df.groupby("description"):
    if len(group) < MIN_OCCURRENCES:
        continue
    if not is_monthly(group):
        continue
    if not is_consistent_amount(group):
        continue

    avg_cost = abs(group["amount"].mean())
    last_paid = group["date"].max()
    months_active = len(group)

    key = merchant.lower()
    difficulty, cancel_note = CANCEL_DIFFICULTY.get(key, ("unknown", "No data on cancellation process."))

    subscriptions.append({
        "name": merchant,
        "monthly_cost": round(avg_cost, 2),
        "yearly_cost": round(avg_cost * 12, 2),
        "months_tracked": months_active,
        "last_payment": last_paid.strftime("%Y-%m-%d"),
        "cancel_difficulty": difficulty,
        "cancel_note": cancel_note,
    })

leaks = [s for s in subscriptions if s["monthly_cost"] >= LEAK_THRESHOLD]

sub_names = {s["name"] for s in subscriptions}
one_offs = df[~df["description"].isin(sub_names)].copy()

total_monthly = sum(s["monthly_cost"] for s in subscriptions)
total_yearly = total_monthly * 12
leak_monthly = sum(s["monthly_cost"] for s in leaks)
leak_yearly = leak_monthly * 12

LEAK_QUIPS = {
    "gym": "{name}-₹{cost}/month. Do you actually go?",
    "netflix": "{name}-₹{cost}/month. Still watching?",
    "spotify": "{name}-₹{cost}/month. Hope it's worth it"
}
DEFAULT_QUIP = "You've been paying ₹{cost}/month for {name}. Sure about that?"

print("SUBSCRIPTION LEAK DETECTOR")

print(f"\nDetected {len(subscriptions)} subscription(s):\n")
for s in sorted(subscriptions, key=lambda x: -x["monthly_cost"]):
    print(f"  • {s['name']:<12} ₹{s['monthly_cost']:>6}/mo  (₹{s['yearly_cost']:>7}/yr)  Cancel: {s['cancel_difficulty']}")

print(f"\nLeaks (above ₹{LEAK_THRESHOLD}/month):\n")
if leaks:
    for s in sorted(leaks, key=lambda x: -x["monthly_cost"]):
        key = s["name"].lower()
        quip = LEAK_QUIPS.get(key, DEFAULT_QUIP)
        print(f"  {quip.format(cost=int(s['monthly_cost']), name=s['name'])}")
        print(f"     {s['cancel_note']}\n")
else:
    print("No leaks found.")

if not one_offs.empty:
    print("One-off transactions:\n")
    for _, row in one_offs.iterrows():
        print(f"  • {row['description']:<12} ₹{abs(row['amount']):<6}  on {row['date'].strftime('%Y-%m-%d')}")

print()
print("─" * 52)
print(f"  Monthly subscription total : ₹{total_monthly:,.0f}")
print(f"  Yearly  subscription total : ₹{total_yearly:,.0f}")
print(f"  Monthly leak total         : ₹{leak_monthly:,.0f}")
print(f"  Estimated yearly waste     : ₹{leak_yearly:,.0f}")
print("─" * 52)

SUBSCRIPTION LEAK DETECTOR

Detected 3 subscription(s):

  • Gym          ₹ 999.0/mo  (₹11988.0/yr)  Cancel: hard
  • Netflix      ₹ 499.0/mo  (₹ 5988.0/yr)  Cancel: medium
  • Spotify      ₹ 119.0/mo  (₹ 1428.0/yr)  Cancel: easy

Leaks (above ₹300/month):

  Gym-₹999/month. Do you actually go?
     Requires in person visit.

  Netflix-₹499/month. Still watching?
     App cancellation works but buried under 3 menus.

One-off transactions:

  • Swiggy       ₹450.0   on 2024-03-15
  • Amazon       ₹1499.0  on 2024-03-18

────────────────────────────────────────────────────
  Monthly subscription total : ₹1,617
  Yearly  subscription total : ₹19,404
  Monthly leak total         : ₹1,498
  Estimated yearly waste     : ₹17,976
────────────────────────────────────────────────────
